# DPE weighting rules — Kaggle / Colab

**Kaggle:** import this notebook, attach your original H2 results ZIP as an input dataset, enable **Internet**, and choose **Run All**. CPU is sufficient; no GPU is required.

**Colab:** open this notebook and choose **Run all**. If the original H2 ZIP is not already in the session, an upload prompt will appear once. Download the results ZIP at the end.

The notebook clones the experiment branch when needed, installs missing dependencies, finds/extracts the H2 archive, infers its track/seed, and runs all weighting rules on the same cached predictions. It does not repeat backbone training, feature extraction or head training. Keep the original H2 ZIP: summary-only reports cannot replace it.

B fits weights; C selects each family's configuration and the overall configuration before test probabilities are read. Uniform is always a candidate. Reusing an already inspected seed-0 test split is exploratory. See the [method instructions](../src/methods/dpe_head_reweighting/README.md#multiple-weighting-rules-without-retraining).


In [ ]:
# Defaults normally need no editing. SOURCE accepts a ZIP or extracted run directory.
SOURCE = None
SUITE_NAME = None  # Automatically inferred; identical reruns reuse verified output.

# Optional overrides for local/offline use or multiple attached archives.
REPO_PATH = None
WORK_DIR = None
SEARCH_PATHS = None
REPO_URL = "https://github.com/chautanphat/waterbirds-shortcut-mitigation.git"
REPO_REF = "experiment/dpe-multiple-weighting-rules"


In [ ]:
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

working = Path.cwd().resolve()
PLATFORM = "kaggle" if Path("/kaggle/working").is_dir() else "colab" if Path("/content").is_dir() else "local"
candidates = [working, *working.parents, working / "waterbirds-shortcut-mitigation"]
REPO = Path(REPO_PATH).expanduser().resolve() if REPO_PATH else next(
    (p for p in candidates if (p / "src/methods/dpe_head_reweighting/notebook_setup.py").is_file()), None)
if WORK_DIR is None:
    base = Path("/kaggle/working") if PLATFORM == "kaggle" else Path("/content") if PLATFORM == "colab" else (REPO / "outputs" if REPO else working)
    WORK_DIR = base / "dpe_weighting_rules_session"
WORK_DIR = Path(WORK_DIR).expanduser().resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
if REPO is None:
    REPO = WORK_DIR / "repository"
    if not REPO.exists():
        try:
            subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", REPO_REF,
                            REPO_URL, str(REPO)], check=True)
        except subprocess.CalledProcessError as error:
            raise RuntimeError("Repository download failed. On Kaggle, enable Internet; alternatively supply REPO_PATH to an updated checkout.") from error
if not (REPO / "src/methods/dpe_head_reweighting/notebook_setup.py").is_file():
    raise FileNotFoundError("REPO_PATH must contain the updated multi-rule implementation. Use the experiment/dpe-multiple-weighting-rules branch.")
missing = [package for module, package in [
    ("numpy", "numpy>=1.26,<3"), ("scipy", "scipy>=1.11,<2"),
    ("torch", "torch>=2.6,<3"), ("yaml", "PyYAML>=6,<7"), ("pandas", "pandas>=2.2,<3")]
    if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import json
import pandas as pd
from IPython.display import display, FileLink
from src.methods.dpe_head_reweighting.rule_suite import select, evaluate
from src.methods.dpe_head_reweighting.notebook_setup import prepare_inputs
print("Platform:", PLATFORM, "| Repository:", REPO)
print("Writable workspace:", WORK_DIR)


## 1. Find the original run and prepare the comparison

One attached H2 ZIP or extracted run is selected automatically. Duplicate ZIP/extracted copies of the same manifest count as one run. If multiple different runs are found, set only `SOURCE` in the first cell to the desired ZIP/directory and rerun. Source files are read only; extraction and results use writable session storage.

The fixed YAML tests uniform, 12 original power configurations, softmax beta 10/30, top-k 10/5, and joint lambda 1/0.1/0.01: **20 candidate configurations**. Candidate order breaks complete C ties after preferring uniform. Settings are frozen before test evaluation.


In [ ]:
CONFIG_PATH = REPO / "configs/dpe_head_reweighting/weighting_rules.yaml"
if SEARCH_PATHS is None:
    SEARCH_PATHS = [REPO / "outputs/dpe_head_reweighting_runs", WORK_DIR / "source-inputs"]
    if PLATFORM == "kaggle":
        SEARCH_PATHS += [Path("/kaggle/input")]
    elif PLATFORM == "colab":
        SEARCH_PATHS += list(Path("/content").glob("*.zip")) + [Path("/content/uploads")]
    else:
        SEARCH_PATHS += list(working.glob("*.zip"))

def upload_source():
    from google.colab import files
    uploads = WORK_DIR / "uploads"
    uploads.mkdir(exist_ok=True)
    previous = Path.cwd()
    try:
        os.chdir(uploads)
        print("Upload the original H2 results ZIP (for example h2-standard-seed0.zip).")
        uploaded = files.upload()
        return [uploads / name for name in uploaded]
    finally:
        os.chdir(previous)

SOURCE_RUN_DIR, OUTPUT_DIR = prepare_inputs(
    SOURCE, SEARCH_PATHS, WORK_DIR, CONFIG_PATH, SUITE_NAME,
    upload=upload_source if PLATFORM == "colab" else None)
print("Read-only source:", SOURCE_RUN_DIR)
print("Separate output:", OUTPUT_DIR)
print(CONFIG_PATH.read_text())


## 2. Fit on B and freeze selection on C

All rules reuse one set of B/C probabilities. Joint weighting minimizes maximum group mean log loss on B plus squared distance from uniform, with nonnegative weights summing to one. A failed optimizer stops the suite rather than silently dropping a candidate.

This stage does not require or open test predictions. Re-running checks saved selection against source hashes, code, settings and recomputed B/C fits. It rejects changed inputs instead of overwriting a completed experiment.


In [ ]:
selection = select(SOURCE_RUN_DIR, CONFIG_PATH, OUTPUT_DIR)
print("Source track/seed:", selection["source_track"], selection["source_seed"])
print("C-selected overall configuration:", selection["selected"])
print("C-selected configuration by family:", selection["per_family_selected"])
comparison = pd.DataFrame([{
    "candidate": row["id"], "rule": row["family"],
    "C WGA": row["metrics"]["wga"], "C accuracy": row["metrics"]["overall"],
    "effective heads": row["effective_heads"],
    "changed C predictions": row["c_prediction_changes"],
    "maximum C probability change": row["c_probability_max_abs_delta"],
} for row in selection["candidates"]])
display(comparison)
weights = pd.DataFrame({row["id"]: row["weights"] for row in selection["candidates"]},
                       index=selection["head_ids"])
display(weights)


## 3. Evaluate the frozen family winners

Only the uniform control, C-selected winner for each family, and C-selected overall winner are evaluated. The table is a comparison, not permission to choose a method using test WGA. If a family retains uniform, its identical results are valid.

The source test cache must share the ensemble identity and have no validation ID overlap. Selection and test-cache hashes bind saved results. Keep already inspected seed-0 findings exploratory and freeze the protocol before later seeds.


In [ ]:
result = evaluate(SOURCE_RUN_DIR, CONFIG_PATH, OUTPUT_DIR)
test_table = pd.DataFrame([{
    "rule": family, "C-selected candidate": row["candidate"],
    "test accuracy": row["metrics"]["overall"], "test WGA": row["metrics"]["wga"],
    "WGA change (pp)": row["delta_pp"]["wga"],
    "accuracy change (pp)": row["delta_pp"]["overall"],
    "changed test predictions": row["changed_predictions"],
    "effective heads": row["effective_heads"],
} for family, row in result["results"].items()])
display(test_table)
group_table = pd.DataFrame([{
    "rule": family, "group": group, "accuracy": accuracy,
    "count": row["metrics"]["counts"][group],
} for family, row in result["results"].items() for group, accuracy in row["metrics"]["groups"].items()])
display(group_table)
print("Report:", OUTPUT_DIR / "report.md")


## 4. Export and download

The results ZIP contains all C candidate scores, exact weights, solver diagnostics, selected test metrics, group counts, paired differences and input/code identities. Keep the original H2 ZIP separately; this ZIP does not duplicate the heads or prediction caches.

Use the download link below on Kaggle. Colab also starts a browser download. To compare another seed, attach/upload that seed's original H2 ZIP and rerun; track, seed and output name are inferred automatically. New input/code/grid identities get a new output directory, while identical reruns reuse verified results.


In [ ]:
import shutil
archive = Path(shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR))
print("Suite archive:", archive)
# A link relative to the notebook's initial directory works with Jupyter/Kaggle.
previous = Path.cwd()
try:
    os.chdir(working)
    display(FileLink(os.path.relpath(archive, working), result_html_prefix="Download results: "))
finally:
    os.chdir(previous)
if PLATFORM == "colab":
    from google.colab import files
    try:
        files.download(str(archive))
    except Exception as error:
        print("Automatic download could not start. Use the link above or the Colab Files panel:", error)
print("Retain the original H2 source ZIP separately.")
